# 04 · Serialization & Aliases: Controlling Input and Output

**Module 15 – Pydantic: Data Validation & Settings** · Estimated time: 50 min

### 🎯 What you will learn
- `model_dump()` / `model_dump_json()` options: `mode`, `include`, `exclude`, `exclude_none`, `exclude_unset`, `exclude_defaults`, `by_alias`, `round_trip`
- Field-level output control: `Field(exclude=True)`, `@field_serializer`, `PlainSerializer` / `WrapSerializer` in `Annotated`
- Whole-model output: `@model_serializer`
- **Aliases**: `alias`, `validation_alias`, `serialization_alias`, `AliasChoices`, `AliasPath`
- **Alias generators** (snake_case ↔ camelCase) and `validate_by_name` / `serialize_by_alias`
- Partial updates (PATCH) with `exclude_unset`

In [1]:
from datetime import datetime, date, timezone
from decimal import Decimal
from typing import Annotated, Any
from pydantic import (BaseModel, ConfigDict, Field, SecretStr, field_serializer, model_serializer,
                      PlainSerializer, AliasChoices, AliasPath, AliasGenerator)
from pydantic.alias_generators import to_camel, to_snake

## 1. `model_dump` options

In [2]:
class Item(BaseModel):
    name: str
    price: Decimal
    released: date
    discount: float | None = None
    tags: list[str] = []
    internal_note: str = Field(default="", exclude=True)       # never serialised

item = Item(name="Pen", price="12.50", released="2026-10-01", tags=["office"], internal_note="supplier X")

print(item.model_dump())                          # Python objects (Decimal, date)
print(item.model_dump(mode="json"))               # JSON-safe: strings/numbers
print(item.model_dump_json(indent=2))
print(item.model_dump(include={"name", "price"}))
print(item.model_dump(exclude_none=True))         # drop None values
print(item.model_dump(exclude_defaults=True))     # drop values equal to their defaults
print(item.model_dump(exclude_unset=True))        # drop fields not explicitly provided

{'name': 'Pen', 'price': Decimal('12.50'), 'released': datetime.date(2026, 10, 1), 'discount': None, 'tags': ['office']}
{'name': 'Pen', 'price': '12.50', 'released': '2026-10-01', 'discount': None, 'tags': ['office']}
{
  "name": "Pen",
  "price": "12.50",
  "released": "2026-10-01",
  "discount": null,
  "tags": [
    "office"
  ]
}
{'name': 'Pen', 'price': Decimal('12.50')}
{'name': 'Pen', 'price': Decimal('12.50'), 'released': datetime.date(2026, 10, 1), 'tags': ['office']}
{'name': 'Pen', 'price': Decimal('12.50'), 'released': datetime.date(2026, 10, 1), 'tags': ['office']}
{'name': 'Pen', 'price': Decimal('12.50'), 'released': datetime.date(2026, 10, 1), 'tags': ['office']}


### Partial updates (HTTP PATCH) with `exclude_unset`

In [3]:
class ItemUpdate(BaseModel):
    name: str | None = None
    price: Decimal | None = None
    discount: float | None = None

patch = ItemUpdate.model_validate({"discount": None, "price": "9.99"})   # user explicitly sets discount to null
changes = patch.model_dump(exclude_unset=True)
print(changes)                                    # name is absent; discount=None IS present
updated = item.model_copy(update=changes)
print(updated)

{'price': Decimal('9.99'), 'discount': None}
name='Pen' price=Decimal('9.99') released=datetime.date(2026, 10, 1) discount=None tags=['office'] internal_note='supplier X'


`exclude_unset` distinguishes "not sent" from "sent as null"; `model_fields_set` shows which fields were provided:

In [4]:
print(patch.model_fields_set)

{'price', 'discount'}


Nested include/exclude use dicts: `model_dump(include={"user": {"name"}, "items": {0: {"price"}}})`.

## 2. Customising how a field is serialised

### `@field_serializer`

In [5]:
class Payment(BaseModel):
    amount: Decimal
    paid_at: datetime
    card_number: str
    api_key: SecretStr

    @field_serializer("amount")
    def money(self, v: Decimal) -> str:
        return f"₹{v:,.2f}"

    @field_serializer("paid_at")
    def iso_utc(self, v: datetime) -> str:
        return v.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

    @field_serializer("card_number", when_used="json")      # only in JSON output
    def mask(self, v: str) -> str:
        return "**** **** **** " + v[-4:]

p = Payment(amount="1234567.5", paid_at="2026-10-20T14:30:00+05:30", card_number="4111111111111234", api_key="sk-123")
print(p.model_dump())
print(p.model_dump_json())

{'amount': '₹1,234,567.50', 'paid_at': '2026-10-20T09:00:00Z', 'card_number': '4111111111111234', 'api_key': SecretStr('**********')}
{"amount":"₹1,234,567.50","paid_at":"2026-10-20T09:00:00Z","card_number":"**** **** **** 1234","api_key":"**********"}


`SecretStr` serialises as `'**********'` by default, so secrets don't leak into logs or JSON.

### Reusable serializers with `Annotated`

In [6]:
Percent = Annotated[float, PlainSerializer(lambda v: f"{v:.0%}", return_type=str)]
UnixTime = Annotated[datetime, PlainSerializer(lambda v: int(v.timestamp()), return_type=int, when_used="json")]

class Stats(BaseModel):
    success_rate: Percent
    updated: UnixTime

s = Stats(success_rate=0.873, updated="2026-10-20T00:00:00Z")
print(s.model_dump(), s.model_dump_json())

{'success_rate': '87%', 'updated': datetime.datetime(2026, 10, 20, 0, 0, tzinfo=TzInfo(0))} {"success_rate":"87%","updated":1792454400}


## 3. `@model_serializer`: full control

In [7]:
class Point(BaseModel):
    x: float
    y: float

    @model_serializer
    def as_list(self) -> list[float]:            # serialise as [x, y] instead of {"x":..,"y":..}
        return [self.x, self.y]

class Route(BaseModel):
    name: str
    points: list[Point]

print(Route(name="walk", points=[{"x": 0, "y": 0}, {"x": 3, "y": 4}]).model_dump_json())

{"name":"walk","points":[[0.0,0.0],[3.0,4.0]]}


`@model_serializer(mode="wrap")` receives a `handler` to get the default dict and then modify it (e.g. add a `"type"` key).

## 4. Aliases: different names outside and inside

| Setting | Used for |
|---|---|
| `alias="x"` | both input **and** (with `by_alias=True`) output |
| `validation_alias=...` | input only (can be `AliasChoices` / `AliasPath`) |
| `serialization_alias="x"` | output only |

In [8]:
class GitHubRepo(BaseModel):
    full_name: str
    stars: int = Field(validation_alias="stargazers_count", serialization_alias="starCount")
    owner_login: str = Field(validation_alias=AliasPath("owner", "login"))          # dig into nested input
    license_name: str | None = Field(default=None,
                                     validation_alias=AliasChoices("license_name", AliasPath("license", "name")))

api_response = {
    "full_name": "python/cpython",
    "stargazers_count": 70123,
    "owner": {"login": "python", "id": 1525981},
    "license": {"key": "other", "name": "Python Software Foundation License"},
}
repo = GitHubRepo.model_validate(api_response)
print(repo)
print(repo.model_dump(by_alias=True))

full_name='python/cpython' stars=70123 owner_login='python' license_name='Python Software Foundation License'
{'full_name': 'python/cpython', 'starCount': 70123, 'owner_login': 'python', 'license_name': 'Python Software Foundation License'}


## 5. Alias generators: snake_case in Python, camelCase in JSON

In [9]:
class APIModel(BaseModel):
    model_config = ConfigDict(
        alias_generator=to_camel,         # first_name ↔ firstName
        validate_by_name=True,            # also accept the Python names on input
        validate_by_alias=True,
        serialize_by_alias=True,          # output aliases by default (2.11+)
    )

class Customer(APIModel):
    first_name: str
    last_name: str
    date_of_birth: date
    is_active: bool = True

c = Customer.model_validate({"firstName": "Asha", "lastName": "Rao", "dateOfBirth": "1995-07-19"})
print(c.first_name, "|", c.model_dump_json())
print(Customer(first_name="Ravi", last_name="K", date_of_birth="1990-01-01").model_dump())   # by name works too

Asha | {"firstName":"Asha","lastName":"Rao","dateOfBirth":"1995-07-19","isActive":true}
{'firstName': 'Ravi', 'lastName': 'K', 'dateOfBirth': datetime.date(1990, 1, 1), 'isActive': True}


Different generators for input and output:

In [10]:
class Mixed(BaseModel):
    model_config = ConfigDict(alias_generator=AliasGenerator(validation_alias=to_camel,
                                                             serialization_alias=lambda n: n.upper()))
    order_id: int

print(Mixed.model_validate({"orderId": 5}).model_dump(by_alias=True))
print(to_camel("created_at_utc"), to_snake("createdAtUtc"))

{'ORDER_ID': 5}
createdAtUtc created_at_utc


> ⚠️ **Changed in 2.11:** `populate_by_name=True` is superseded by `validate_by_name=True` (+ `validate_by_alias`); `serialize_by_alias` lets you default to alias output without passing `by_alias=True` everywhere.

## 6. Round-tripping

In [11]:
from pydantic import Json

class Envelope(BaseModel):
    payload: Json[dict[str, int]]

e = Envelope(payload='{"a": 1}')
print(e.model_dump(), "|", e.model_dump_json(round_trip=True))   # round_trip: output can be validated again
print(Envelope.model_validate_json(e.model_dump_json(round_trip=True)))

{'payload': {'a': 1}} | {"payload":"{\"a\":1}"}
payload={'a': 1}


## 📝 Summary
- `model_dump(mode="json")` / `model_dump_json()` with `include`, `exclude`, `exclude_none/unset/defaults`, `by_alias`.
- `exclude_unset` + `model_fields_set` implement PATCH semantics correctly.
- Customise output with `Field(exclude=True)`, `@field_serializer`, `PlainSerializer` in `Annotated`, `@model_serializer`.
- Aliases map external names: `validation_alias` (+ `AliasChoices`/`AliasPath`), `serialization_alias`, `alias_generator=to_camel`.

## 🏋️ Practice
1. Model a JSON API that uses camelCase keys; accept snake_case too; output camelCase.
2. A `User` model with `password: SecretStr` and `email`; make sure `model_dump_json()` never contains the password, and add `@field_serializer` to mask the email as `a***@example.com`.
3. Write `apply_patch(model, patch_model)` that updates only the fields explicitly sent.